In [2]:
# %pip install -q "langchain>=1.0.0" langchain-openai langchain-anthropic langchain-google-genai langchain-text-splitters tiktoken python-dotenv langchain-community faiss-cpu

In [1]:
import os

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv

# .env 로드
load_dotenv()


# API Key 확인
print(
    "[OK] OpenAI API 키 설정됨"
    if os.getenv("OPENAI_API_KEY")
    else "[ERROR] OpenAI API 키 미설정"
)

print(
    "[OK] LangSmith API 키 설정됨"
    if os.getenv("LANGSMITH_API_KEY")
    else "[ERROR] LangSmith API 키 미설정"
)

print(
    "[OK] Gemini API 키 설정됨"
    if os.getenv("GEMINI_API_KEY")
    else "[ERROR] Gemini API 키 미설정"
)


# LangChain 버전 확인
print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

[OK] OpenAI API 키 설정됨
[OK] LangSmith API 키 설정됨
[OK] Gemini API 키 설정됨
LangChain: 1.4.0
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


In [ ]:
from langchain.chat_models import init_chat_model

gpt_model = init_chat_model("gpt-4o-mini", temperature=0.2, max_tokens=2048)
gemini_model = init_chat_model( "gemini-3.5-flash-lite", model_provider="google_genai")


### 순차 체인

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

translate_prompt = ChatPromptTemplate.from_template(
    "다음 한국어를 영어로 번역하세요: {korean_word}"
)

explain_prompt = ChatPromptTemplate.from_template(
    "다음 영어 단어를 한국어로 자세히 설명하세요: {english_word}"
)


## 순차 체인
chain1 = translate_prompt | gpt_model | StrOutputParser()

chain2 = (
    {"english_word" : chain1}
    | explain_prompt
    | gpt_model
    | StrOutputParser()
)

result = chain2.invoke({"korean_word": "인공지능"})
print(result)

"인공지능"은 영어로 "artificial intelligence"라고 하며, 이는 인간의 지능을 모방하거나 재현하는 컴퓨터 시스템이나 프로그램을 의미합니다. 인공지능은 기계가 학습, 추론, 문제 해결, 이해 및 언어 처리와 같은 지능적인 작업을 수행할 수 있도록 하는 기술입니다.

인공지능은 크게 두 가지로 나눌 수 있습니다. 첫째, 좁은 인공지능(Narrow AI)은 특정 작업에 특화된 시스템으로, 예를 들어 음성 인식, 이미지 인식, 게임 플레이 등 특정 분야에서 뛰어난 성능을 발휘합니다. 둘째, 일반 인공지능(General AI)은 인간과 유사한 수준의 지능을 갖춘 시스템으로, 다양한 작업을 수행할 수 있는 능력을 지니고 있습니다. 현재까지는 일반 인공지능은 이론적인 개념에 가깝고, 대부분의 연구와 개발은 좁은 인공지능에 집중되고 있습니다.

인공지능의 발전은 머신러닝, 딥러닝, 자연어 처리(NLP)와 같은 다양한 기술을 통해 이루어지며, 이는 데이터 분석과 패턴 인식을 기반으로 합니다. 인공지능은 의료, 금융, 자동차, 제조업 등 여러 분야에서 활용되고 있으며, 우리의 일상생활에도 점점 더 많은 영향을 미치고 있습니다. 예를 들어, 스마트폰의 음성 비서, 추천 시스템, 자율주행차 등이 인공지능의 응용 사례입니다. 

결론적으로, 인공지능은 기술적 혁신을 통해 인간의 삶을 개선하고, 다양한 문제를 해결하는 데 기여하는 중요한 분야입니다.


### 다단계 순차 체인

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

analyze_prompt = ChatPromptTemplate.from_template(
    "다음 주제의 핵심 키워드 3개를 추출하세요: {topic}"
)

outline_prompt = ChatPromptTemplate.from_template(
    """다음 키워드를 바탕으로 글의 개요를 작성하세요:
    키워드: {keywords}
    원본 주제: {topic}
    """
    )

conent_prompt = ChatPromptTemplate.from_template(
    """
    다음 개요를 바탕으로 300자 내외의 글을 작성하세요 :
    개요 : {outline}
    """
)

chain = (
    {"topic" : RunnablePassthrough()} 
    | RunnablePassthrough.assign(keywords = analyze_prompt | gpt_model | StrOutputParser())
    | RunnablePassthrough.assign(outline = outline_prompt | gpt_model | StrOutputParser())
    | conent_prompt
    | gpt_model
    | StrOutputParser()
)

result = chain.invoke("기후 변화와 지속 가능한 발전")
print(result)

기후 변화는 지구의 기온 상승과 극단적인 기상 현상으로 나타나며, 이는 주로 인간의 산업화와 도시화에 기인합니다. 이러한 변화는 생태계의 생물 다양성을 감소시키고, 인간 사회의 식량 안보와 건강에 심각한 위협을 가합니다. 지속 가능한 발전은 경제적, 사회적, 환경적 측면을 통합하여 미래 세대를 위한 자원을 보존하는 것을 목표로 합니다. 이를 위해 국제 사회는 파리 협정과 같은 노력을 기울이고 있으며, 개인과 공동체의 참여가 필수적입니다. 재생 가능 에너지의 활용과 자원 효율성 증대는 기후 변화 완화의 중요한 전략입니다. 따라서 기후 변화 대응은 긴급하며, 지속 가능한 발전을 위한 공동의 노력이 필요합니다. 환경 보호는 우리의 미래를 지키는 길임을 잊지 말아야 합니다.


### 병렬 체인

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

positive_prompt = ChatPromptTemplate.from_template(
    "{topic}의 긍정적인 측면 3가지를 한 단어로 설명하세요."
)

negative_prompt = ChatPromptTemplate.from_template(
    "{topic}의 부정적인 측면 3가지를 한 단어로 설명하세요."
)

neutral_prompt = ChatPromptTemplate.from_template(
    "{topic}에 대한 객관적인 현황을 한 단어로 설명하세요."
)

parallel_chain = RunnableParallel(
    positive=positive_prompt | gpt_model | StrOutputParser(),
    negative=negative_prompt | gpt_model | StrOutputParser(),
    neutral=neutral_prompt | gpt_model | StrOutputParser(),
)

results = parallel_chain.invoke({"topic": "원격 근무"})

print("긍정적 측면 :")
print(results["positive"])
print("\n부정적 측면 :")
print(results["negative"])
print("\n객관적 현황 :")
print(results["neutral"])

# 병렬 결과 통합
systhesis_prompt = ChatPromptTemplate.from_template(
    """다음 분석을 종합하여 한문장으로 작성하세요:
        긍정:
        {positive}
        
        단점:
        {negative}
        
        객관적:
        {neutral}
        
        균형 잡힌 결론을 1문장으로 작성하세요.
    """
)

full_chain = (
    parallel_chain
    | systhesis_prompt
    | gpt_model
    | StrOutputParser()
)

results = full_chain.invoke({"topic": "원격 근무"})
print(results)


긍정적 측면 :
유연성, 생산성, 편안함.

부정적 측면 :
고립감, 소통, 집중력.

객관적 현황 :
유연성.
유연성은 생산성과 편안함을 높이는 긍정적인 요소지만, 고립감과 소통의 어려움, 집중력 저하와 같은 단점도 동반할 수 있어 균형 있는 접근이 필요하다.


### 조건부 분기

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch


delivery_prompt = ChatPromptTemplate.from_template(
    "고객의 배송 문의에 답변하세요: {question}"
)

refund_prompt = ChatPromptTemplate.from_template(
    "고객의 환불 문의에 답변하세요: {question}"
)

default_prompt = ChatPromptTemplate.from_template(
    "상담원 연결로 안내하세요: {question}"
)

def check_request(input_dict):
    question = input_dict["question"]
    
    if "배송" in question:
        return "delivery"
    elif "환불" in question:
        return "refund"
    else:
        return "default"
    
branch_chain = RunnableBranch(
    (lambda x: check_request(x) == "delivery", delivery_prompt | gpt_model | StrOutputParser()),
    (lambda x: check_request(x) == "refund", refund_prompt | gpt_model | StrOutputParser()),
    default_prompt | gpt_model | StrOutputParser()
)

result1 = branch_chain.invoke({"question" : "배송 예정일을 알고싶어요."})
result2 = branch_chain.invoke({"question" : "환불 규정을 알고싶어요."})
result3 = branch_chain.invoke({"question" : "주문을 수정하고싶어요."})

print(result1)
print("------------------")
print(result2)
print("------------------")
print(result3)

안녕하세요! 고객님, 배송 문의 주셔서 감사합니다. 배송 예정일은 주문하신 상품의 종류와 배송 지역에 따라 다를 수 있습니다. 일반적으로 주문 확인 후 2-5일 이내에 배송이 이루어집니다. 정확한 배송일을 확인하시려면 주문 번호를 알려주시면 더욱 자세한 안내를 드릴 수 있습니다. 감사합니다!
------------------
안녕하세요! 고객님, 환불 규정에 대해 문의 주셔서 감사합니다.

저희의 환불 규정은 다음과 같습니다:

1. **환불 요청 기간**: 제품 수령 후 14일 이내에 환불 요청을 하셔야 합니다.
2. **제품 상태**: 환불 요청 시 제품이 사용되지 않고 원래 포장 상태여야 합니다.
3. **환불 절차**: 고객센터에 연락 주시면 환불 절차를 안내해 드리겠습니다. 필요한 정보를 제공해 주시면 신속하게 처리해 드리겠습니다.
4. **환불 방법**: 환불은 원래 결제 수단으로 진행됩니다.

추가적인 질문이 있으시면 언제든지 문의해 주세요. 감사합니다!
------------------
죄송하지만, 직접 상담원과 연결해 드릴 수는 없습니다. 하지만 주문 수정에 대한 일반적인 안내를 드릴 수 있습니다. 보통 주문 수정은 해당 쇼핑몰이나 서비스의 고객센터에 연락하시거나, 웹사이트의 '주문 내역'에서 수정할 수 있는 옵션을 찾아보시면 됩니다. 추가적인 도움이 필요하시면 어떤 부분에서 어려움을 겪고 계신지 말씀해 주세요.


### 함수 기반 라우팅

In [28]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

def route_by_topic(input_dict):
    topic = input_dict.get("topic","").lower()
    
    if "code" in topic or "programming" in topic:
        return "technical"
    elif "business" in topic or "consultant" in topic:
        return "business"
    else:
        return "general"
    
# 주제별 체인 생성
chains = {
    "technical" : ChatPromptTemplate.from_template(
        "기술 전문가로서 답변: {question}") | gpt_model | StrOutputParser(),
    "business" : ChatPromptTemplate.from_template(
        "비즈니스 컨설턴트로서 답변: {question}") | gpt_model | StrOutputParser(),
    "general" : ChatPromptTemplate.from_template(
        "일반적 관점에서 답변: {question}") | gpt_model | StrOutputParser(),
}

def routing_chain(input_dict):
    # 어디로 보낼지 결정
    route = route_by_topic(input_dict)
    # 결정된 곳의 체인을 가져와서 실행
    return chains[route].invoke(input_dict)

router = RunnableLambda(routing_chain)

result = router.invoke({
    "topic" : "LangChain",
    "question" : "효율적인 작성 방법은?"
})

print(result)

    
    

효율적인 작성 방법은 여러 가지 요소를 고려해야 합니다. 다음은 일반적인 관점에서 효율적인 글쓰기를 위한 몇 가지 팁입니다:

1. **목적 설정**: 글을 쓰기 전에 무엇을 전달하고 싶은지 명확히 하세요. 목표가 분명하면 글의 방향성이 확립됩니다.

2. **계획 세우기**: 글의 구조를 미리 계획하세요. 서론, 본론, 결론의 형식을 따르거나, 주제에 맞는 아웃라인을 작성하는 것이 좋습니다.

3. **자료 조사**: 필요한 정보를 수집하고, 신뢰할 수 있는 출처에서 자료를 확보하세요. 이는 글의 신뢰성을 높이는 데 도움이 됩니다.

4. **시간 관리**: 일정한 시간 동안 집중해서 글을 쓰고, 그 후에는 잠시 휴식을 취하세요. Pomodoro 기법(25분 집중, 5분 휴식 등)을 활용할 수 있습니다.

5. **초안 작성**: 처음부터 완벽한 글을 쓰려고 하지 말고, 일단 초안을 작성하세요. 수정은 나중에 하면 됩니다.

6. **간결한 표현**: 불필요한 단어나 문장을 줄이고, 명확하고 간결하게 표현하세요. 독자가 쉽게 이해할 수 있도록 하는 것이 중요합니다.

7. **피드백 받기**: 다른 사람에게 글을 읽어보게 하고 피드백을 받으세요. 다양한 시각에서의 의견은 글을 개선하는 데 큰 도움이 됩니다.

8. **수정 및 교정**: 초안이 완성되면 여러 번 읽어보며 수정하세요. 문법, 철자, 문장 구조 등을 점검하여 최종적으로 완성도를 높입니다.

9. **정기적인 연습**: 글쓰기는 연습을 통해 발전합니다. 정기적으로 글을 쓰고, 다양한 주제에 도전해 보세요.

이러한 방법들을 통해 효율적으로 글을 작성할 수 있습니다. 각자의 스타일에 맞게 조정해보세요!


### 복합 패턴

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


# 검색할 자료
docs= [
    Document(page_content="LangChain은 LLM 애플리케이션을 만들기 위한 프레임워크입니다."),
    Document(page_content="RAG는 외부 문서를 검색한 뒤 그 정보를 LLM에게 제공하는 방식입니다."),
    Document(page_content="RunnableBranch는 조건에 따라 서로 다른 체인을 실행합니다."),
]

# 문장을 숫자로 바꿔주는 도구(벡터화)
embeddings = OpenAIEmbeddings()

# 검색할 수 있도록 벡터화된 문서들을 가지고 있는 FAISS 저장소
# 이 문서들을 이 임베딩 방식으로 정리해서 FAISS 검색창고 하나 만들어줘.
vectorstore = FAISS.from_documents(docs,embeddings)

In [30]:
# 입력, 출력 str 타입 힌트
def retrieve_context(query: str) -> str:
    # LangChain의 FAISS 연동 클래스가 제공하는 .similarity_search 유사도 검색 기능
    docs = vectorstore.similarity_search(query, k=1)
    
    return "\n".join(
        doc.page_content
        for doc in docs
    )

In [31]:
rag_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        context=RunnableLambda(
            lambda x: retrieve_context(x["question"])
        )
    )

    | ChatPromptTemplate.from_template(
        """
컨텍스트를 참고하여 질문에 답변하세요.

컨텍스트:
{context}

질문:
{question}

답변:
"""
    )
    | gpt_model
    | StrOutputParser()
)

result = rag_chain.invoke({
    "question": "LangChain이란?"
})

print(result)

LangChain은 LLM(대형 언어 모델) 애플리케이션을 개발하기 위한 프레임워크입니다. 이 프레임워크는 다양한 기능과 도구를 제공하여 개발자들이 LLM을 활용한 애플리케이션을 보다 쉽게 구축하고 관리할 수 있도록 돕습니다.


### 다중 모델 앙상블

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

prompt = ChatPromptTemplate.from_template("{question}")

ensemble_chain = RunnableParallel(
    gpt    = prompt | gpt_model | StrOutputParser(),
    gemini = prompt | gemini_model | StrOutputParser(),
)

synthesis_prompt = ChatPromptTemplate.from_template(
    """두 AI의 응답을 비교하고 최선의 답변을 종합하세요.
    
    # GPT 응답:
    {gpt}
    
    # Gemini 응답:
    {gemini}
    
    # 종합 답변:
    """
)

responses = ensemble_chain.invoke({"question": "효과적인 AI 학습 방법은?, 20자 내외로 말해주세요."})

print("GPT 답변:")
print(responses["gpt"])

print("\nGemini 답변:")
print(responses["gemini"])

final_result = (
    synthesis_prompt
    | gpt_model
    | StrOutputParser()
).invoke(responses)

print("\n종합 답변:")
print(final_result)



GPT 답변:
다양한 데이터 활용과 반복 학습이 중요합니다.

Gemini 답변:
명확한 목적 설정과 양질의 데이터 활용 (19자)

종합 답변:
효과적인 AI 개발을 위해서는 명확한 목적 설정과 다양한 양질의 데이터 활용, 그리고 반복 학습이 중요합니다.


### RunnablePassthrough

- 입력 데이터 보존

In [ ]:
# 질문 데이터 보존

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(
        summary = ChatPromptTemplate.from_template("{text}를 요약") | gpt_model | StrOutputParser() 
    )
    | RunnablePassthrough.assign(
        keywords = ChatPromptTemplate.from_template("{text}의 키워드") | gpt_model | StrOutputParser()
    )
)

result = chain.invoke({"text": "긴 문서 내용..."})

- 데이터 변환

In [ ]:
from langchain_core.runnables import RunnableLambda

def preprocess(input_dict):
    return{
        "processed_text" : input_dict["text"].strip().lower(),
        "original"       : input_dict["text"]
    }
    
def postprocess(output):
    return{
        "result": output,
    }
    
chain = (
    RunnableLambda(preprocess)
    | ChatPromptTemplate.from_template("분석: {processd_text}")
    | gpt_model
    | StrOutputParser()
    | RunnableLambda(postprocess)
)


first=RunnableLambda(preprocess) middle=[ChatPromptTemplate(input_variables=['processd_text'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['processd_text'], input_types={}, partial_variables={}, template='분석: {processd_text}'), additional_kwargs={})]), ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_messa

### LangChain Runnable 요약

| 패턴 | 코드 | 설명 |
|---|---|---|
| 순차 | `A \| B` | A → B 순서대로 |
| 병렬 | `RunnableParallel()` | A, B 동시에 |
| 분기 | `RunnableBranch()` | 조건에 따라 선택 |
| 데이터 보존 | `RunnablePassthrough.assign()` | 기존 데이터 + 추가 |
| 데이터 변환 | `RunnableLambda()` | 내 함수로 데이터 변경 |

### 멀티 체인 설계 팁

- 병렬 실행 → 여러 API를 동시에 호출할 때 `RunnableParallel()` 사용
- 결과 보존 → 중간 결과를 남길 때 `assign()` 사용
- 에러 처리 → 오류에 대비해 `with_fallbacks()` 사용
- 복잡한 로직 → 하나의 체인에 몰아넣지 말고 여러 체인으로 나누기

